In [1]:
!pip install -q -U langchain langchain-groq langchain-community langchain-huggingface faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.5 MB/s eta 0:00:00


In [4]:
import os
from google.colab import userdata

In [5]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [6]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [7]:
# LLM & CHAT MODEL
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.5)

In [8]:
#  PROMPTS & SIMPLE CHAIN (LCEL)
prompt = PromptTemplate.from_template("Explain {topic} to a {audience}.")
chain = prompt | llm | StrOutputParser()

In [9]:
# 3: DOCUMENT LOADERS & VECTOR STORES (RAG)
docs = [
    Document(page_content="LangChain was launched in 2022 by Harrison Chase."),
    Document(page_content="It is designed to simplify the creation of LLM-powered applications.")
]
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# 4: AGENTS & TOOLS
@tool
def calculate_checkout_total(quantity: int, price_per_unit: float) -> str:
    """Calculates final price including a fixed shipping fee."""
    return f"Total: ₹{(quantity * price_per_unit) + 97}"

tools = [calculate_checkout_total]
llm_with_tools = llm.bind_tools(tools)

In [11]:
# 5: MEMORY
print("1. Chain Output:", chain.invoke({"topic": "RAG", "audience": "student"}))
print("2. Vector Store Query:", vectorstore.similarity_search("Who created LangChain?")[0].page_content)

1. Chain Output: RAG (Red, Amber, Green) is a system used to track student progress, particularly in the UK. It's a simple and visual way to show how well you're doing in your studies. Here's how it works:

1. **Red**: This color indicates that you're struggling or falling behind in a subject. It might mean you need extra help or support to get back on track.
2. **Amber**: This color shows that you're making progress, but there are some areas where you need to improve. You might need to work a bit harder or focus on specific skills to reach your targets.
3. **Green**: This color means you're doing well and meeting your targets. You're on track to achieve your goals, and you should be proud of yourself!

The RAG system is often used to:

* Set targets and track progress
* Identify areas where you need extra support
* Encourage you to work hard and improve
* Celebrate your successes and achievements

Your teachers might use the RAG system to give you feedback on your work, and you might 